# Brain Dance - De3DGS Video Processing

Process your own video with **Deformable 3D Gaussians (De3DGS)** on Google Colab.

**Version**: 2.0.0  
**Date**: 2026-02-10  
**Requirements**: Colab (T4/L4 GPU), CUDA 12.8, 12GB+ VRAM  
**Estimated Runtime**: 30-60 minutes depending on video length

## Workflow

1. **Section 0**: Configuration
2. **Section A**: Environment Setup (GPU, dependencies, CUDA kernels)
3. **Section B**: Upload Your Video
4. **Section C**: Process Video with De3DGS
5. **Section D**: Download Results
6. **Section E**: Summary

---
## Section 0: Configuration

In [ ]:
# Cell 0.1: Configuration
"""
All tunable parameters in one place.
Modify these values to customize your run.
"""

from dataclasses import dataclass

@dataclass
class NotebookConfig:
    """All tunable parameters in one place."""
    # Repository
    repo_url: str = "https://github.com/ujseah/brain-dance.git"
    branch: str = "feat/de3dgs-migration"
    
    # Training
    training_iterations: int = 5000  # Reduced for validation (full: 20000)
    checkpoint_interval: int = 1000
    
    # VRAM limit
    max_vram_gb: float = 12.0
    
    # Paths
    checkpoint_dir: str = "/content/brain_dance_checkpoint"
    output_dir: str = "/content/brain_dance_output"
    repo_dir: str = "/content/brain-dance"

CONFIG = NotebookConfig()
print(f"Configuration loaded:")
print(f"  Repository: {CONFIG.repo_url}")
print(f"  Branch: {CONFIG.branch}")
print(f"  Training iterations: {CONFIG.training_iterations}")

In [ ]:
# Cell 0.2: Resume Detection
"""
Checkpoint manager for resuming after Colab disconnects.
If you want a fresh run, call CHECKPOINT.reset()
"""

import os
import json
from pathlib import Path
from datetime import datetime

class CheckpointManager:
    """Manage notebook execution checkpoints for resume capability."""
    
    def __init__(self, checkpoint_dir: str):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.state_file = self.checkpoint_dir / "state.json"
        self.state = self._load_state()
    
    def _load_state(self) -> dict:
        if self.state_file.exists():
            return json.loads(self.state_file.read_text())
        return {"completed_sections": [], "start_time": None}
    
    def _save_state(self):
        self.state_file.write_text(json.dumps(self.state, indent=2))
    
    def is_complete(self, section: str) -> bool:
        return section in self.state["completed_sections"]
    
    def mark_complete(self, section: str):
        if section not in self.state["completed_sections"]:
            self.state["completed_sections"].append(section)
            self._save_state()
            print(f"Checkpoint saved: {section}")
    
    def reset(self):
        """Clear all checkpoints (for fresh run)."""
        self.state = {"completed_sections": [], "start_time": None}
        self._save_state()
        print("Checkpoints cleared - starting fresh")

CHECKPOINT = CheckpointManager(CONFIG.checkpoint_dir)
print(f"Checkpoint status: {len(CHECKPOINT.state['completed_sections'])} sections complete")
if CHECKPOINT.state['completed_sections']:
    print(f"  Completed: {CHECKPOINT.state['completed_sections']}")
    print("  To start fresh, run: CHECKPOINT.reset()")

---
## Section A: Environment Setup

In [ ]:
# Cell A.1: GPU Verification
import torch

def get_gpu_info() -> dict:
    """Get detailed GPU information."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "NO GPU DETECTED!\n\n"
            "To fix this:\n"
            "1. Go to Runtime > Change runtime type\n"
            "2. Select 'T4 GPU' or 'L4 GPU' under Hardware accelerator\n"
            "3. Click Save and re-run this cell"
        )
    
    props = torch.cuda.get_device_properties(0)
    
    # Detect GPU tier for runtime estimation
    gpu_name = props.name.lower()
    if "a100" in gpu_name:
        tier = "A100"
        estimated_time = "10-15 min"
    elif "l4" in gpu_name:
        tier = "L4"
        estimated_time = "15-25 min"
    elif "v100" in gpu_name:
        tier = "V100"
        estimated_time = "15-20 min"
    elif "t4" in gpu_name:
        tier = "T4"
        estimated_time = "25-35 min"
    else:
        tier = "Unknown"
        estimated_time = "30-45 min"
    
    return {
        "name": props.name,
        "tier": tier,
        "vram_gb": props.total_memory / 1e9,
        "cuda_version": torch.version.cuda,
        "pytorch_version": torch.__version__,
        "estimated_time": estimated_time,
    }

gpu_info = get_gpu_info()
print("=" * 50)
print("GPU VERIFICATION")
print("=" * 50)
print(f"[OK] GPU: {gpu_info['name']}")
print(f"[OK] Tier: {gpu_info['tier']}")
print(f"[OK] VRAM: {gpu_info['vram_gb']:.1f} GB")
print(f"[OK] CUDA: {gpu_info['cuda_version']}")
print(f"[OK] PyTorch: {gpu_info['pytorch_version']}")
print(f"[TIME] Estimated processing time: {gpu_info['estimated_time']}")

if gpu_info['vram_gb'] < CONFIG.max_vram_gb:
    print(f"\n[WARN] VRAM ({gpu_info['vram_gb']:.1f} GB) is below recommended {CONFIG.max_vram_gb} GB")

CHECKPOINT.mark_complete("section_a1_gpu")

In [ ]:
# Cell A.2: Clone Repository
import subprocess
import time

def clone_with_retry(url: str, dest: str, branch: str, max_retries: int = 3):
    """Clone repository with exponential backoff retry."""
    for attempt in range(max_retries):
        try:
            if os.path.exists(dest):
                print(f"Repository already exists at {dest}")
                result = subprocess.run(
                    ["git", "-C", dest, "remote", "get-url", "origin"],
                    capture_output=True, text=True
                )
                if url in result.stdout:
                    print("Updating existing repository...")
                    subprocess.run(["git", "-C", dest, "fetch", "--all"], check=True)
                    subprocess.run(["git", "-C", dest, "checkout", branch], check=True)
                    subprocess.run(["git", "-C", dest, "pull", "--ff-only"], check=True)
                    subprocess.run(["git", "-C", dest, "submodule", "update", "--init", "--recursive"], check=True)
                    return True
                else:
                    print("Different repository exists, removing...")
                    subprocess.run(["rm", "-rf", dest], check=True)
            
            print(f"Cloning {url} (attempt {attempt + 1}/{max_retries})...")
            subprocess.run([
                "git", "clone", "--recursive",
                "-b", branch,
                url, dest
            ], check=True)
            return True
        
        except subprocess.CalledProcessError as e:
            wait_time = 2 ** attempt
            print(f"Clone failed, retrying in {wait_time}s...")
            time.sleep(wait_time)
    
    raise RuntimeError(f"Failed to clone repository after {max_retries} attempts")

if CHECKPOINT.is_complete("section_a2_clone"):
    print("[SKIP] Section A.2 already complete, skipping clone...")
else:
    clone_with_retry(
        CONFIG.repo_url,
        CONFIG.repo_dir,
        CONFIG.branch
    )
    os.chdir(CONFIG.repo_dir)
    
    if not os.path.exists("deformable3dgs/train.py"):
        raise RuntimeError("De3DGS submodule not properly initialized!")
    
    print("[OK] Repository cloned and verified")
    CHECKPOINT.mark_complete("section_a2_clone")

In [ ]:
# Cell A.3: Install Dependencies (CUDA 12.8 compatible)
if CHECKPOINT.is_complete("section_a3_deps"):
    print("[SKIP] Section A.3 already complete, skipping dependency install...")
else:
    print("Installing PyTorch with CUDA 12.8 support...")
    !pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
    
    print("\nInstalling other dependencies...")
    !pip install -q -r {CONFIG.repo_dir}/notebooks/requirements-colab.txt --ignore-installed torch torchvision
    
    print("[OK] Dependencies installed")
    CHECKPOINT.mark_complete("section_a3_deps")

In [ ]:
# Cell A.4: Compile De3DGS CUDA Kernels
import subprocess

if CHECKPOINT.is_complete("section_a4_cuda"):
    print("[SKIP] Section A.4 already complete, skipping CUDA compilation...")
else:
    print("Compiling De3DGS CUDA kernels...")
    print("This may take 5-10 minutes on first run.\n")
    
    setup_script = f"{CONFIG.repo_dir}/scripts/setup_de3dgs.sh"
    
    if os.path.exists(setup_script):
        # Run with proper error checking
        result = subprocess.run(
            ["bash", setup_script],
            cwd=CONFIG.repo_dir,
            capture_output=False  # Show output in real-time
        )
        
        if result.returncode != 0:
            raise RuntimeError(
                f"CUDA kernel compilation failed with exit code {result.returncode}\n"
                "Check the output above for details."
            )
    else:
        # Manual compilation fallback
        print("Setup script not found, compiling manually...")
        de3dgs_dir = f"{CONFIG.repo_dir}/deformable3dgs"
        
        result1 = subprocess.run(
            ["pip", "install", "-e", f"{de3dgs_dir}/submodules/diff-gaussian-rasterization"],
            capture_output=False
        )
        if result1.returncode != 0:
            raise RuntimeError("diff-gaussian-rasterization compilation failed")
        
        result2 = subprocess.run(
            ["pip", "install", "-e", f"{de3dgs_dir}/submodules/simple-knn"],
            capture_output=False
        )
        if result2.returncode != 0:
            raise RuntimeError("simple-knn compilation failed")
    
    print("[OK] CUDA kernels compiled")
    CHECKPOINT.mark_complete("section_a4_cuda")

In [ ]:
# Cell A.5: Verify Installation
print("=" * 50)
print("INSTALLATION VERIFICATION")
print("=" * 50)

errors = 0

try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print("[OK] diff-gaussian-rasterization")
except ImportError as e:
    print(f"[FAIL] diff-gaussian-rasterization: {e}")
    errors += 1

try:
    from simple_knn import _C
    print("[OK] simple-knn")
except ImportError as e:
    print(f"[FAIL] simple-knn: {e}")
    errors += 1

try:
    from plyfile import PlyData
    print("[OK] plyfile")
except ImportError as e:
    print(f"[FAIL] plyfile: {e}")
    errors += 1

if errors == 0:
    print("\n[OK] All installations verified!")
    CHECKPOINT.mark_complete("section_a5_verify")
else:
    raise RuntimeError(f"Installation verification failed with {errors} error(s)")

---
## Section B: Upload Your Video

In [ ]:
# Cell B.1: Upload Video
"""
Upload your video to process with De3DGS.

Supported formats: MP4, MOV, AVI, WEBM
Recommended: MP4 with H.264 codec, 720p-1080p, 2-10 seconds

Tips for best results:
- Keep the video short (2-10 seconds)
- Use smooth camera motion
- Avoid fast motion blur
- Good lighting helps quality
"""
from google.colab import files
import subprocess

USER_VIDEO_DIR = f"{CONFIG.output_dir}/user_video"
USER_VIDEO_PATH = None

print("=" * 50)
print("VIDEO UPLOAD")
print("=" * 50)
print("\nSupported formats: MP4, MOV, AVI, WEBM")
print("Recommended: MP4 with H.264 codec, 720p-1080p, 2-10 seconds\n")

uploaded = files.upload()

if uploaded:
    os.makedirs(USER_VIDEO_DIR, exist_ok=True)
    
    for filename, data in uploaded.items():
        video_path = f"{USER_VIDEO_DIR}/{filename}"
        with open(video_path, 'wb') as f:
            f.write(data)
        
        # Get video info
        result = subprocess.run([
            'ffprobe', '-v', 'quiet', '-print_format', 'json',
            '-show_format', '-show_streams', video_path
        ], capture_output=True, text=True)
        
        if result.returncode == 0:
            import json as json_lib
            info = json_lib.loads(result.stdout)
            duration = float(info['format'].get('duration', 0))
            video_stream = next((s for s in info['streams'] if s['codec_type'] == 'video'), {})
            width = video_stream.get('width', 'unknown')
            height = video_stream.get('height', 'unknown')
            fps = eval(video_stream.get('r_frame_rate', '30/1'))
            
            print(f"\n[OK] Video uploaded: {filename}")
            print(f"     Resolution: {width}x{height}")
            print(f"     Duration: {duration:.1f} seconds")
            print(f"     Frame rate: {fps:.1f} fps")
            print(f"     Estimated frames: {int(duration * fps)}")
            
            USER_VIDEO_PATH = video_path
        else:
            print(f"[WARN] Could not read video metadata for {filename}")
            USER_VIDEO_PATH = video_path
        
        # Display first frame preview
        from IPython.display import display, Image as IPImage
        preview_path = f"{USER_VIDEO_DIR}/preview.jpg"
        subprocess.run([
            'ffmpeg', '-y', '-i', video_path, '-vframes', '1',
            '-q:v', '2', preview_path
        ], capture_output=True)
        
        if os.path.exists(preview_path):
            print("\nFirst frame preview:")
            display(IPImage(filename=preview_path, width=400))
else:
    print("[WARN] No video uploaded. Please upload a video to continue.")
    USER_VIDEO_PATH = None

---
## Section C: Process Video with De3DGS

In [ ]:
# Cell C.1: Extract Frames and Prepare Dataset
"""
Extract frames from your video and prepare for De3DGS training.
This step converts your video into the format De3DGS expects.
"""
import subprocess
from pathlib import Path

if USER_VIDEO_PATH is None:
    raise RuntimeError("No video uploaded! Please run Section B first.")

FRAMES_DIR = f"{CONFIG.output_dir}/frames"
os.makedirs(FRAMES_DIR, exist_ok=True)

print("=" * 50)
print("EXTRACTING FRAMES")
print("=" * 50)

# Extract frames at original fps
result = subprocess.run([
    'ffmpeg', '-y', '-i', USER_VIDEO_PATH,
    '-qscale:v', '2',
    f'{FRAMES_DIR}/frame_%04d.png'
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"[FAIL] Frame extraction failed: {result.stderr}")
    raise RuntimeError("Frame extraction failed")

frames = sorted(Path(FRAMES_DIR).glob("*.png"))
print(f"[OK] Extracted {len(frames)} frames to {FRAMES_DIR}")

# Display sample frames
import matplotlib.pyplot as plt
from PIL import Image

if len(frames) >= 3:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    indices = [0, len(frames)//2, -1]
    for ax, idx in zip(axes, indices):
        img = Image.open(frames[idx])
        ax.imshow(img)
        ax.set_title(f"Frame {idx if idx >= 0 else len(frames)+idx}")
        ax.axis('off')
    plt.suptitle("Sample Frames from Your Video")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell C.2: Run COLMAP for Camera Poses (if needed)
"""
COLMAP extracts camera poses from your video frames.
This is required for De3DGS to understand the 3D structure.

Note: This step can take 10-30 minutes depending on video length.
"""
import subprocess

COLMAP_OUTPUT = f"{CONFIG.output_dir}/colmap"
os.makedirs(COLMAP_OUTPUT, exist_ok=True)

print("=" * 50)
print("RUNNING COLMAP")
print("=" * 50)
print("\nThis step extracts camera poses from your video.")
print("It may take 10-30 minutes depending on video length.\n")

# Check if COLMAP is available
colmap_check = subprocess.run(["which", "colmap"], capture_output=True)
if colmap_check.returncode != 0:
    print("Installing COLMAP...")
    !apt-get update -qq && apt-get install -qq colmap

# Run COLMAP
convert_script = f"{CONFIG.repo_dir}/deformable3dgs/convert.py"

if os.path.exists(convert_script):
    !python {convert_script} -s {FRAMES_DIR} --no_gpu
else:
    print("[INFO] Running COLMAP manually...")
    # Feature extraction
    !colmap feature_extractor \
        --database_path {COLMAP_OUTPUT}/database.db \
        --image_path {FRAMES_DIR} \
        --ImageReader.single_camera 1
    
    # Feature matching
    !colmap exhaustive_matcher \
        --database_path {COLMAP_OUTPUT}/database.db
    
    # Sparse reconstruction
    sparse_dir = f"{COLMAP_OUTPUT}/sparse/0"
    os.makedirs(sparse_dir, exist_ok=True)
    !colmap mapper \
        --database_path {COLMAP_OUTPUT}/database.db \
        --image_path {FRAMES_DIR} \
        --output_path {COLMAP_OUTPUT}/sparse

print("\n[OK] COLMAP processing complete")

In [ ]:
# Cell C.3: Train De3DGS on Your Video
"""
Train Deformable 3D Gaussians on your video.
This creates a 4D representation of your scene.
"""
import time

DE3DGS_DIR = f"{CONFIG.repo_dir}/deformable3dgs"
TRAINING_OUTPUT = f"{CONFIG.output_dir}/training"

print("=" * 50)
print("TRAINING De3DGS")
print("=" * 50)
print(f"\nIterations: {CONFIG.training_iterations}")
print(f"Output: {TRAINING_OUTPUT}")
print(f"Estimated time: {gpu_info['estimated_time']}\n")

start_time = time.time()

# Determine input path (COLMAP output or frames dir with transforms)
input_path = FRAMES_DIR
if os.path.exists(f"{COLMAP_OUTPUT}/sparse/0"):
    input_path = COLMAP_OUTPUT

# Run training
!cd {DE3DGS_DIR} && python train.py \
    -s {input_path} \
    -m {TRAINING_OUTPUT} \
    --iterations {CONFIG.training_iterations}

elapsed = time.time() - start_time
print(f"\n[OK] Training completed in {elapsed/60:.1f} minutes")

In [ ]:
# Cell C.4: Render Results
"""
Render the trained De3DGS model into an MP4 video.
"""
from pathlib import Path

RENDER_OUTPUT = f"{CONFIG.output_dir}/renders"
os.makedirs(RENDER_OUTPUT, exist_ok=True)

print("=" * 50)
print("RENDERING RESULTS")
print("=" * 50)

training_output = Path(TRAINING_OUTPUT)

if not training_output.exists():
    print("[SKIP] No training output found. Run Cell C.3 first.")
else:
    # Run De3DGS renderer
    render_script = f"{CONFIG.repo_dir}/deformable3dgs/render.py"
    
    if os.path.exists(render_script):
        print("\nRendering frames...")
        !cd {CONFIG.repo_dir}/deformable3dgs && python render.py \
            -m {training_output}
        
        # Find rendered frames
        render_dir = training_output / "train" / f"ours_{CONFIG.training_iterations}" / "renders"
        if not render_dir.exists():
            render_dir = training_output / "test" / f"ours_{CONFIG.training_iterations}"
        
        if render_dir.exists():
            render_frames = sorted(render_dir.glob("*.png"))
            
            if render_frames:
                print(f"\n[OK] Rendered {len(render_frames)} frames")
                
                # Create MP4
                mp4_path = f"{RENDER_OUTPUT}/de3dgs_render.mp4"
                !ffmpeg -y -framerate 15 \
                    -pattern_type glob -i '{render_dir}/*.png' \
                    -c:v libx264 -pix_fmt yuv420p \
                    -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" \
                    {mp4_path} 2>/dev/null
                
                if os.path.exists(mp4_path):
                    mp4_size = os.path.getsize(mp4_path) / 1e6
                    print(f"[OK] Rendered video: {mp4_path} ({mp4_size:.1f} MB)")
                    
                    from IPython.display import Video, display
                    print("\nPlaying rendered video:")
                    display(Video(mp4_path, embed=True, width=640))
            else:
                print("[WARN] No rendered frames found")
        else:
            print(f"[WARN] Render directory not found")
    else:
        print("[INFO] Render script not found")

---
## Section D: Download Results

In [ ]:
# Cell D.1: Package Results into ZIP
"""
Package PLY files, MP4 videos, and model checkpoints
into a single ZIP file for download.
"""
import zipfile
from pathlib import Path

DOWNLOAD_ZIP = "/content/brain_dance_results.zip"

print("=" * 50)
print("PACKAGING RESULTS")
print("=" * 50)

files_added = 0

with zipfile.ZipFile(DOWNLOAD_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    
    # Add PLY files
    print("\nAdding PLY files...")
    for ply_file in Path(CONFIG.output_dir).rglob("*.ply"):
        arcname = f"ply_files/{ply_file.parent.name}/{ply_file.name}"
        zf.write(ply_file, arcname)
        files_added += 1
        if files_added <= 5:
            print(f"  + {arcname}")
    if files_added > 5:
        print(f"  ... and {files_added - 5} more PLY files")
    
    # Add MP4 renders
    print("\nAdding MP4 videos...")
    mp4_count = 0
    for mp4_file in Path(CONFIG.output_dir).rglob("*.mp4"):
        arcname = f"videos/{mp4_file.name}"
        zf.write(mp4_file, arcname)
        files_added += 1
        mp4_count += 1
        print(f"  + {arcname} ({os.path.getsize(mp4_file) / 1e6:.1f} MB)")
    
    if mp4_count == 0:
        print("  (No MP4 files found)")
    
    # Add model checkpoints
    print("\nAdding model checkpoints...")
    pth_count = 0
    for pth_file in Path(CONFIG.output_dir).rglob("*.pth"):
        arcname = f"models/{pth_file.parent.name}/{pth_file.name}"
        zf.write(pth_file, arcname)
        files_added += 1
        pth_count += 1
        print(f"  + {arcname} ({os.path.getsize(pth_file) / 1e6:.1f} MB)")
    
    if pth_count == 0:
        print("  (No .pth files found)")

zip_size_mb = os.path.getsize(DOWNLOAD_ZIP) / 1e6
print(f"\n" + "=" * 50)
print(f"[OK] Created: {DOWNLOAD_ZIP}")
print(f"     Total files: {files_added}")
print(f"     Size: {zip_size_mb:.1f} MB")
print("=" * 50)

In [ ]:
# Cell D.2: Download Results
"""
Download the packaged ZIP file to your local machine.
Your browser will prompt you to save the file.
"""
from google.colab import files

print("=" * 50)
print("DOWNLOAD RESULTS")
print("=" * 50)
print()

if os.path.exists(DOWNLOAD_ZIP):
    zip_size = os.path.getsize(DOWNLOAD_ZIP) / 1e6
    print(f"Downloading: brain_dance_results.zip")
    print(f"Size: {zip_size:.1f} MB")
    print()
    print("Your browser will prompt you to save the file...")
    print()
    
    files.download(DOWNLOAD_ZIP)
    
    print("\n[OK] Download initiated!")
    print()
    print("ZIP contents:")
    print("  ply_files/  - PLY Gaussian files")
    print("  videos/     - Rendered MP4 videos")
    print("  models/     - Model checkpoints (.pth)")
else:
    print("[FAIL] No results ZIP found.")
    print("       Run Cell D.1 first to package the results.")

---
## Section E: Summary

In [ ]:
# Cell E.1: Final Summary
"""
Display final processing summary.
"""
import torch

print("=" * 60)
print("DE3DGS PROCESSING SUMMARY")
print("=" * 60)
print()

# Check completed sections
env_ok = CHECKPOINT.is_complete("section_a5_verify")
print(f"Environment Setup:  {'PASS' if env_ok else 'FAIL'}")

video_ok = USER_VIDEO_PATH is not None
print(f"Video Uploaded:     {'PASS' if video_ok else 'SKIP'}")

training_ok = os.path.exists(f"{CONFIG.output_dir}/training")
print(f"Training Complete:  {'PASS' if training_ok else 'SKIP'}")

results_ok = os.path.exists(DOWNLOAD_ZIP)
print(f"Results Packaged:   {'PASS' if results_ok else 'SKIP'}")

print()

# Memory usage
if torch.cuda.is_available():
    max_allocated = torch.cuda.max_memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Peak GPU Memory: {max_allocated:.1f} GB / {total:.1f} GB ({max_allocated/total*100:.0f}%)")

print()
print("=" * 60)
if env_ok and video_ok and training_ok and results_ok:
    print("PROCESSING COMPLETE - Download your results from Section D")
else:
    print("PROCESSING INCOMPLETE - Check the sections above")
print("=" * 60)